In [47]:
 
from azure.identity import InteractiveBrowserCredential

cred = InteractiveBrowserCredential(tenant_id="8aefdf9f-8780-46bf-8fb7-4c924653a8be")

token = cred.get_token("https://storage.azure.com/.default")

print(token)

 

INFO Request URL: 'https://login.microsoftonline.com/8aefdf9f-8780-46bf-8fb7-4c924653a8be/v2.0/.well-known/openid-configuration'
Request method: 'GET'
Request headers:
    'User-Agent': 'azsdk-python-identity/1.25.1 Python/3.11.5 (Windows-10-10.0.22631-SP0)'
No body was attached to the request
INFO Response status: 200
Response headers:
    'Cache-Control': 'max-age=86400, private'
    'Content-Type': 'application/json; charset=utf-8'
    'Strict-Transport-Security': 'REDACTED'
    'X-Content-Type-Options': 'REDACTED'
    'Access-Control-Allow-Origin': 'REDACTED'
    'Access-Control-Allow-Methods': 'REDACTED'
    'P3P': 'REDACTED'
    'x-ms-request-id': '3a8d4e7a-f8da-411d-9dcd-1a0065000800'
    'x-ms-ests-server': 'REDACTED'
    'x-ms-srs': 'REDACTED'
    'Content-Security-Policy-Report-Only': 'REDACTED'
    'X-XSS-Protection': 'REDACTED'
    'Set-Cookie': 'REDACTED'
    'Date': 'Wed, 19 Nov 2025 16:19:40 GMT'
    'Content-Length': '1996'
INFO Request URL: 'https://login.microsoftonli

AccessToken(token='eyJ0eXAiOiJKV1QiLCJhbGciOiJSUzI1NiIsIng1dCI6InJ0c0ZULWItN0x1WTdEVlllU05LY0lKN1ZuYyIsImtpZCI6InJ0c0ZULWItN0x1WTdEVlllU05LY0lKN1ZuYyJ9.eyJhdWQiOiJodHRwczovL3N0b3JhZ2UuYXp1cmUuY29tIiwiaXNzIjoiaHR0cHM6Ly9zdHMud2luZG93cy5uZXQvOGFlZmRmOWYtODc4MC00NmJmLThmYjctNGM5MjQ2NTNhOGJlLyIsImlhdCI6MTc2MzU2ODg4NSwibmJmIjoxNzYzNTY4ODg1LCJleHAiOjE3NjM1NzM4MTIsImFjciI6IjEiLCJhaW8iOiJBVVFBdS84YUFBQUFIRFhqeDFVUElSUFpvb2UyUDBIbHJaVitlbWZzZGhlYkVtaHdUaEJseGt0UWFyOE9haGYwUTAzNG51VDFreVp5M2tqUnJtck9QNXQ0NXNNWGJBQzcwdz09IiwiYW1yIjpbInB3ZCIsInJzYSJdLCJhcHBpZCI6IjA0YjA3Nzk1LThkZGItNDYxYS1iYmVlLTAyZjllMWJmN2I0NiIsImFwcGlkYWNyIjoiMCIsImRldmljZWlkIjoiMjE1OWE4NGMtYjIyZC00NGMzLTgwNzktMDhlMGM2MjdlZWM4IiwiZmFtaWx5X25hbWUiOiJLaWFuaSIsImdpdmVuX25hbWUiOiJBbWlyaG9zc2VpbiIsImdyb3VwcyI6WyJiYzY1MzIwMC1jZjRlLTRkZTUtOGQwMS03NTEyNDIxOTMxMGMiLCI2MjNlMmUwYS1kMDE1LTQyYmMtODk5Yi1hNzYxNmM3OTQ0MjQiLCJjZjYxN2ExMC1lM2M0LTRlOTctOThlNi02NjZiMWYwMTkyOTEiLCJlM2ZmNGUxMS0xZmM0LTQxYTctYmNmOC1kMTJiMDAwYzRhZmYiLCIyOTdlMjUxMi1jMGQ4

# Fusing the detection vehicles info

In [2]:
from pathlib import Path
import sys
import os
import django
from django.conf import settings


project_root = Path(r".\backend")
sys.path.insert(0, str(project_root))

os.environ.setdefault("DJANGO_SETTINGS_MODULE", "backend.processor.settings")

django.setup()

print("Loaded settings module:", os.environ.get("DJANGO_SETTINGS_MODULE"))
print("DATABASES:", settings.DATABASES)


Loaded settings module: backend.processor.settings
DATABASES: {'default': {'ENGINE': 'django.db.backends.sqlite3', 'NAME': WindowsPath('C:/MyApps/VisionSaver/backend/db.sqlite3'), 'TEST': {'MIRROR': 'default', 'CHARSET': None, 'COLLATION': None, 'MIGRATE': True, 'NAME': None}, 'ATOMIC_REQUESTS': False, 'AUTOCOMMIT': True, 'CONN_MAX_AGE': 0, 'CONN_HEALTH_CHECKS': False, 'OPTIONS': {}, 'TIME_ZONE': None, 'USER': '', 'PASSWORD': '', 'HOST': '', 'PORT': ''}}


In [50]:
from record.models import Record, RecordLog # type: ignore
from ai.models import AutoDetection, DetectionLines # type: ignore
from asgiref.sync import sync_to_async

detections = await sync_to_async(list)(AutoDetection.objects.all())


In [ ]:
import pandas as pd
movement_to_keys = {"through": 10, "left": 20, "right": 30}


def get_auto_detection_movements(auto_df, lines, record_id):
    min_max = auto_df.groupby("track_id")["time"].agg({"min", "max"})
    min_max["diff"] = min_max["max"] - min_max["min"]
    zone_numbers = {key:len(value) for key, value in lines.items()}
    significant_tracks = min_max[min_max["diff"] > 1].index.tolist()
    significant_auto_df = auto_df[auto_df["track_id"].isin(significant_tracks)]
    automobile_cls_ids = [2, 5, 7]
    significant_auto_df_vehicles = significant_auto_df[significant_auto_df["cls_id"].isin(automobile_cls_ids)]
    significant_auto_df_vehicles["line_index_key"] = significant_auto_df_vehicles["line_index"].apply(
        lambda x: next((movement_to_keys[k] for k in movement_to_keys if k in str(x).lower()), None)
    )
    significant_auto_df_vehicles = significant_auto_df_vehicles.dropna(subset=["line_index_key"])
    significant_auto_df_vehicles["final_turn_movement"] = None
    groups = significant_auto_df_vehicles.groupby("track_id")[["time", "line_index_key", "zone_index"]]
    for track_id, group in groups:
        subgroups = {
            line_key: list(int(z) for z in sub["zone_index"].unique())
            for line_key, sub in group.groupby("line_index_key")
        }
        zone_numbers_mapped = {}
        for line_name, zones_list in lines.items():
            mapped_key = next(
            (movement_to_keys[k] for k in movement_to_keys if k in str(line_name).lower()),
            None,
            )
            if mapped_key is not None:
                zone_numbers_mapped[mapped_key] = len(zones_list)
        
        # replace the string-keyed zone_numbers with numeric-keyed mapping so subsequent checks work
        zone_numbers = zone_numbers_mapped
        for line_key, zones in subgroups.items():
            if len(zones) == zone_numbers.get(line_key, 0):
                # check if zones is incremental, if not we are not interested
                
                if zones == sorted(zones):
                    
                    significant_auto_df_vehicles.loc[significant_auto_df_vehicles["track_id"] == track_id, "final_turn_movement"] = line_key
                    # TODO: we might need to check this value against other line keys. This
                    # might not be the end of the process. Check this later
                    break
    
    significant_auto_df_vehicles.dropna(subset=["final_turn_movement"], inplace=True)
    significant_auto_df_vehicles["record_id"] = record_id
    return significant_auto_df_vehicles

def get_matching_data(manaul_df, auto_df, auto_lines):
    significant_auto_df_vehicles = get_auto_detection_movements(auto_df, auto_lines)
    significant_auto_df_vehicles["matched"] = False
    manual_df_temp = manaul_df.copy()
    manual_df_temp["line_index_key"] = manual_df_temp["turn_movement"].apply(
        lambda x: movement_to_keys.get(str(x).lower(), None)
    )
    manual_df_temp["matched"] = False
    detection_time_df = significant_auto_df_vehicles.groupby(["track_id", "final_turn_movement"]).agg({"time": "max"}).rename({"time": "detection_time"}, axis=1).reset_index()
    for index, row in detection_time_df.iterrows():
        track_id = row["track_id"]
        final_turn_movement = row["final_turn_movement"]
        detection_time = row["detection_time"]
        time_window_start = detection_time - 5  # 2 seconds before
        time_window_end = detection_time + 5    # 2 seconds after
        mask = (
            (manual_df_temp["line_index_key"] == final_turn_movement) &
            (manual_df_temp["time"] >= time_window_start) &
            (manual_df_temp["time"] <= time_window_end) &
            (~manual_df_temp["matched"])
        )
        matched_indices = manual_df_temp[mask].index
        manual_df_temp.loc[matched_indices, "matched"] = True
        significant_auto_df_vehicles.loc[significant_auto_df_vehicles["track_id"] == track_id, "matched"] = True
    return significant_auto_df_vehicles


In [124]:
from tqdm import tqdm
final_df = pd.DataFrame()
starting_number = 1
for detection in tqdm(detections):
    record_id = detection.record_id
    record = await sync_to_async(list)(Record.objects.filter(id=record_id))
    if record:
        record = record[0]
        if record.finished_detecting:
            record_manual_logs = await sync_to_async(list)(RecordLog.objects.filter(record_id=record_id))
            record_manual_df_dict = {"time":[], "turn_movement":[], "record_id": []}

            for log in record_manual_logs:
                record_manual_df_dict["time"].append(log.time)
                record_manual_df_dict["turn_movement"].append(log.turn_movement)
                record_manual_df_dict["record_id"].append(record_id)
            record_manual_df = pd.DataFrame(record_manual_df_dict)
            detection_auto_df = pd.read_csv(detection.file_name)
            detection_lines = await sync_to_async(list)(DetectionLines.objects.filter(record_id=record_id))
            auto_lines = detection_lines[0].lines
            matching_data = get_matching_data(record_manual_df, detection_auto_df, auto_lines)
            # Remap track_id
            unique_track_ids = sorted(matching_data["track_id"].unique())
            track_id_map = {old_id: new_id for new_id, old_id in enumerate(unique_track_ids, start=starting_number)}
            matching_data.loc[:, "track_id"] = matching_data["track_id"].map(track_id_map)
            starting_number += len(unique_track_ids) + 1
            final_df = pd.concat([final_df, matching_data], ignore_index=True)

  3%|▎         | 4/115 [00:00<00:11,  9.37it/s]C:\Users\amki003\AppData\Local\Temp\ipykernel_28996\1243377525.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  significant_auto_df_vehicles["line_index_key"] = significant_auto_df_vehicles["line_index"].apply(
  9%|▊         | 10/115 [00:01<00:13,  7.69it/s]C:\Users\amki003\AppData\Local\Temp\ipykernel_28996\1243377525.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  significant_auto_df_vehicles["line_index_key"] = significant_auto_df_vehicles["line_in

In [126]:
final_df.to_csv("vehicle_movements.csv", index=False)

In [127]:
final_df

,track_id,x1,y1,x2,y2,cls_id,confidence,time,in_area,line_index,zone_index,line_index_key,final_turn_movement,matched
0,1,0.435003,0.221162,0.499184,0.322312,2,0.811245,0.066733,True,through,0,10.0,10.0,True
1,2,0.512382,0.363224,0.606207,0.552886,2,0.477426,0.066733,True,through,0,10.0,10.0,True
2,3,0.556482,0.098199,0.590168,0.145772,2,0.365922,0.066733,True,through,0,10.0,10.0,True
3,1,0.434532,0.221814,0.498774,0.323175,2,0.792563,0.100100,True,through,0,10.0,10.0,True
4,2,0.512257,0.363086,0.606152,0.552892,2,0.470074,0.100100,True,through,0,10.0,10.0,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3140251,10510,0.360873,0.432363,0.479815,0.680138,2,0.748573,893.422000,True,through,0,10.0,10.0,True
3140252,10510,0.357074,0.442372,0.479543,0.698436,2,0.813484,893.455000,True,through,0,10.0,10.0,True
3140253,10510,0.352698,0.454708,0.478715,0.719037,2,0.769498,893.489000,True,through,0,10.0,10.0,True
3140254,10510,0.291467,0.649023,0.453555,1.004230,2,0.476249,893.889000,True,through,1,10.0,10.0,True
